# 09 — Blocked OLS and mixed models

Walkthrough of doekit 0.4 analysis: fixed blocks, HC3 standard errors, lack-of-fit, and REML mixed models.

In [ ]:
import numpy as np
import pandas as pd
import doekit as ed
print("doekit", ed.__version__)

## 1. Fixed blocks

Replicate a $2^2$ factorial across two blocks with a large block shift.

In [ ]:
base = ed.full_factorial({"A": [-1, 1], "B": [-1, 1]})
mat = pd.concat([base.matrix, base.matrix], ignore_index=True)
design = ed.Design(
    matrix=mat,
    factors=list(base.factors),
    model=ed.Model.main_effects(["A", "B"]),
    metadata={"kind": "FullFactorial"},
)
design = ed.attach_blocks(design, [0, 0, 0, 0, 1, 1, 1, 1], name="block")

rng = np.random.default_rng(0)
y = (
    10
    + 2.0 * design.matrix["A"].to_numpy()
    - 1.5 * design.matrix["B"].to_numpy()
    + 5.0 * (design.matrix["block"].to_numpy() == 1).astype(float)
    + rng.normal(0, 0.05, design.n_runs)
)

fit = ed.fit_linear_model(design, y, blocks="block")
print(fit)
print(ed.anova_table(fit))

## 2. Robust SE (HC3)

In [ ]:
pb = ed.plackett_burman(5)
scale = 0.05 + 0.4 * (pb.matrix["factor1"].to_numpy() > 0)
y_het = 1.0 + 2.0 * pb.matrix["factor1"].to_numpy() + rng.normal(0, 1, pb.n_runs) * scale
print(ed.fit_linear_model(pb, y_het, cov_type="nonrobust").summary_frame())
print(ed.fit_linear_model(pb, y_het, cov_type="HC3").summary_frame())

## 3. Lack of fit (center replicates)

In [ ]:
fac = ed.full_factorial({"A": [-1, 1], "B": [-1, 1]})
centers = pd.DataFrame({"A": [0, 0, 0], "B": [0, 0, 0]})
mat2 = pd.concat([fac.matrix, centers], ignore_index=True)
d2 = ed.Design(
    matrix=mat2,
    factors=list(fac.factors),
    model=ed.Model.parse("0 ~ A + B + A:B + A^2 + B^2"),
)
y2 = (
    5
    + 1.5 * d2.matrix["A"]
    - 0.8 * d2.matrix["B"]
    + 0.5 * d2.matrix["A"] ** 2
    + rng.normal(0, 0.1, d2.n_runs)
)
print(ed.lack_of_fit(d2, y2))

## 4. Mixed model (random batch intercept)

In [ ]:
n_g, n_per = 4, 6
n = n_g * n_per
group = np.repeat(np.arange(n_g), n_per)
x = rng.uniform(-1, 1, n)
re = rng.normal(0, 1.5, n_g)
y3 = 2.0 + 1.2 * x + re[group] + rng.normal(0, 0.3, n)
dm = ed.Design(
    matrix=pd.DataFrame({"x": x, "batch": group}),
    factors=[ed.ContinuousFactor("x", -1, 1)],
    model=ed.Model.parse("0 ~ x"),
)
mix = ed.fit_mixed_model(dm, y3, groups="batch")
print(mix)
print("re_var", mix.re_var)
print(mix.to_dict()["schema"])